# Adverbs

In [43]:
import importlib.util
import sys
from pathlib import Path

# Path to your third-version parser
converter_path = Path(r"C:\Users\rahaa\Dropbox\MPCD\syntax_project\corpus\mptf_parser.py")

# Dynamically import the module
spec = importlib.util.spec_from_file_location("mptf_parser", converter_path)
mptf_parser = importlib.util.module_from_spec(spec)
sys.modules["mptf_parser"] = mptf_parser
spec.loader.exec_module(mptf_parser)

# Set input and output folders
input_folder = r"C:\Users\rahaa\Dropbox\MPCD\exports_28-1-2026"
output_folder = r"C:\Users\rahaa\Dropbox\MPCD\conllu_output"

# Convert TSV/CSV files to CoNLL-U and load as Sentence objects
corpus = mptf_parser.parse_corpus(input_folder, output_folder)

# Split into full corpus
my_corpus = corpus

# --- NEW CRITERION: sentence is syntactically annotated if it has exactly one root ---
syntactically_annotated_corpus = [
    s for s in my_corpus
    if sum(1 for t in s.get_tokens() if t.deprel == "root") == 1
]

print(f"✔ Parsed {len(my_corpus)} sentences")
print(f"✔ Syntactically annotated sentences: {len(syntactically_annotated_corpus)}")


--- Starting Corpus Parsing Pipeline ---
Step 1: Converting files from 'C:\Users\rahaa\Dropbox\MPCD\exports_28-1-2026'


Converting source files:   0%|          | 0/46 [00:00<?, ?it/s]


Source file conversion complete!

Step 2: Loading processed files from 'C:\Users\rahaa\Dropbox\MPCD\conllu_output'


Loading and Parsing CoNLL-U files:   0%|          | 0/48 [00:00<?, ?it/s]


Successfully loaded 41461 sentences.

--- Pipeline Complete ---
✔ Parsed 41461 sentences
✔ Syntactically annotated sentences: 5789


In [44]:
import os
from collections import Counter
from conllu import parse
from conllu.parser import DEFAULT_FIELD_PARSERS
import mptf_parser as mptf

# ======================================================
# HELPERS
# ======================================================

def clean_source_filename(filename):
    """
    Remove '_mptf' and file extension from a conllu filename.
    Example: dk4_ch22_mptf.conllu → dk4_ch22
    """
    name = os.path.splitext(filename)[0]
    if name.endswith("_mptf"):
        name = name[:-5]
    return name


# ======================================================
# LOAD CORPORA (NO THRESHOLDS)
# ======================================================

print("\n--- Loading corpora ---")

custom_field_parsers = DEFAULT_FIELD_PARSERS.copy()
custom_field_parsers["id"] = lambda line, i: line[i]
custom_field_parsers["head"] = lambda line, i: line[i]

my_corpus = []
syntactically_annotated_corpus = []

conllu_files = sorted(
    f for f in os.listdir(output_folder)
    if f.lower().endswith(".conllu")
    and "outdated" not in f.lower()
)

for filename in conllu_files:
    file_path = os.path.join(output_folder, filename)
    clean_name = clean_source_filename(filename)

    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = f.read()

    # --- Clean malformed lines ---
    lines = raw_data.splitlines()
    clean_lines = [
        line for line in lines
        if line.startswith("#")
        or line.strip() == ""
        or line.count("\t") == 9
    ]
    clean_data = "\n".join(clean_lines) + "\n"

    # --- Parse sentences ---
    sentences = parse(clean_data, field_parsers=custom_field_parsers)

    for sent in sentences:
        sentence_obj = mptf.Sentence(sent, source_filename=clean_name)
        sentence_obj.metadata["source_filename"] = clean_name
        my_corpus.append(sentence_obj)

        # --- syntactic criterion: exactly one root ---
        num_roots = sum(
            1 for tok in sentence_obj.get_tokens()
            if tok.deprel == "root"
        )

        if num_roots == 1:
            syntactically_annotated_corpus.append(sentence_obj)


# ======================================================
# CONFIRMATION
# ======================================================

print("✔ Corpora loaded successfully.")

print("\nmy_corpus (all texts):")
print(f"  texts:     {len(set(s.metadata['source_filename'] for s in my_corpus))}")
print(f"  sentences: {len(my_corpus)}")

print("\nsyntactically_annotated_corpus (1 root only):")
print(f"  texts:     {len(set(s.metadata['source_filename'] for s in syntactically_annotated_corpus))}")
print(f"  sentences: {len(syntactically_annotated_corpus)}")


# ======================================================
# PER-TEXT STATISTICS
# ======================================================

total_sentences = Counter(
    s.metadata["source_filename"] for s in my_corpus
)

syntactic_sentences = Counter(
    s.metadata["source_filename"] for s in syntactically_annotated_corpus
)

TOTAL_CORPUS_SIZE = len(my_corpus)
SYNTACTIC_CORPUS_SIZE = len(syntactically_annotated_corpus)

print("\n📊 Per-text corpus contributions:")
print(
    f"{'File':<30}"
    f"{'Total':>7}"
    f"{'% of corpus':>12}"
    f"{'Syntactic':>11}"
    f"{'% of syntactic':>15}"
    f"{'Coverage %':>13}"
)
print("-" * 88)

for fname in sorted(total_sentences):
    total = total_sentences[fname]
    syn = syntactic_sentences.get(fname, 0)

    pct_of_corpus = (total / TOTAL_CORPUS_SIZE * 100) if TOTAL_CORPUS_SIZE else 0.0
    pct_of_syn = (syn / SYNTACTIC_CORPUS_SIZE * 100) if SYNTACTIC_CORPUS_SIZE else 0.0
    coverage = (syn / total * 100) if total else 0.0

    print(
        f"{fname:<30}"
        f"{total:>7}"
        f"{pct_of_corpus:>11.2f}%"
        f"{syn:>11}"
        f"{pct_of_syn:>14.2f}%"
        f"{coverage:>12.2f}%"
    )


# ======================================================
# LIST FILES WITH SYNTACTIC ANNOTATION
# ======================================================

annotated_files = sorted(
    set(s.metadata["source_filename"] for s in syntactically_annotated_corpus)
)

print("\n📄 Source files with at least one syntactically annotated sentence:")
for fname in annotated_files:
    print("-", fname)



--- Loading corpora ---
✔ Corpora loaded successfully.

my_corpus (all texts):
  texts:     42
  sentences: 35096

syntactically_annotated_corpus (1 root only):
  texts:     26
  sentences: 5789

📊 Per-text corpus contributions:
File                            Total % of corpus  Syntactic % of syntactic   Coverage %
----------------------------------------------------------------------------------------
AOD-K20                           106       0.30%         67          1.16%       63.21%
Col-J2                              7       0.02%          0          0.00%        0.00%
Col-K20-1                           3       0.01%          0          0.00%        0.00%
Col-K43a-2                          5       0.01%          0          0.00%        0.00%
Col-K5-1                            8       0.02%          0          0.00%        0.00%
Col-K7b-1                           2       0.01%          0          0.00%        0.00%
Col-K7b-2                           8       0.02%         

In [45]:
import csv

# ======================================================
# EXPORT PER-TEXT STATISTICS TO CSV
# ======================================================

csv_path = os.path.join(output_folder, "per_text_corpus_contributions.csv")

with open(csv_path, "w", encoding="utf-8", newline="") as csvfile:
    writer = csv.writer(csvfile)

    # Header
    writer.writerow([
        "file",
        "total_sentences",
        "pct_of_whole_corpus",
        "syntactic_sentences",
        "pct_of_syntactic_corpus",
        "coverage_pct"
    ])

    # Rows
    for fname in sorted(total_sentences):
        total = total_sentences[fname]
        syn = syntactic_sentences.get(fname, 0)

        pct_of_corpus = (total / TOTAL_CORPUS_SIZE * 100) if TOTAL_CORPUS_SIZE else 0.0
        pct_of_syn = (syn / SYNTACTIC_CORPUS_SIZE * 100) if SYNTACTIC_CORPUS_SIZE else 0.0
        coverage = (syn / total * 100) if total else 0.0

        writer.writerow([
            fname,
            total,
            round(pct_of_corpus, 4),
            syn,
            round(pct_of_syn, 4),
            round(coverage, 4)
        ])

print(f"\n✔ CSV exported to: {csv_path}")



✔ CSV exported to: C:\Users\rahaa\Dropbox\MPCD\conllu_output\per_text_corpus_contributions.csv


In [46]:
import os
from conllu import parse
from conllu.parser import DEFAULT_FIELD_PARSERS
import mptf_parser as mptf


# ======================================================
# HELPERS
# ======================================================

def clean_source_filename(filename):
    """
    Remove '_mptf' and file extension from a conllu filename.
    Example: dk4_ch22_mptf.conllu → dk4_ch22
    """
    name = os.path.splitext(filename)[0]   # remove .conllu
    if name.endswith("_mptf"):
        name = name[:-5]
    return name


# ======================================================
# LOAD BOTH CORPORA
# ======================================================

print("\n--- Loading corpora ---")

custom_field_parsers = DEFAULT_FIELD_PARSERS.copy()
custom_field_parsers["id"] = lambda line, i: line[i]
custom_field_parsers["head"] = lambda line, i: line[i]

my_corpus = []
syntactically_annotated_corpus = []

MIN_SENTENCES_SYN = 10  # threshold for syntactic corpus only

conllu_files = sorted(
    f for f in os.listdir(output_folder)
    if f.lower().endswith(".conllu")
    and "outdated" not in f.lower()
)

for filename in conllu_files:
    file_path = os.path.join(output_folder, filename)
    clean_name = clean_source_filename(filename)

    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = f.read()

    # --- Clean malformed lines ---
    lines = raw_data.splitlines()
    clean_lines = [
        line for line in lines
        if line.startswith("#")
        or line.strip() == ""
        or line.count("\t") == 9
    ]
    clean_data = "\n".join(clean_lines) + "\n"

    # --- Parse sentences ---
    sentences = parse(clean_data, field_parsers=custom_field_parsers)

    # --------------------------------------------------
    # Add ALL sentences to FULL corpus
    # --------------------------------------------------
    for sent in sentences:
        sentence_obj = mptf.Sentence(
            sent,
            source_filename=clean_name
        )
        sentence_obj.metadata["source_filename"] = clean_name
        my_corpus.append(sentence_obj)

    # --------------------------------------------------
    # Add sentences to SYNTACTIC corpus only if:
    # 1. Text has ≥ MIN_SENTENCES_SYN
    # 2. Sentence has exactly one root
    # --------------------------------------------------
    if len(sentences) >= MIN_SENTENCES_SYN:
        for sent in sentences:
            sentence_obj = mptf.Sentence(
                sent,
                source_filename=clean_name
            )
            sentence_obj.metadata["source_filename"] = clean_name

            num_roots = sum(
                1 for tok in sentence_obj.get_tokens()
                if tok.deprel == "root"
            )

            if num_roots == 1:
                syntactically_annotated_corpus.append(sentence_obj)


# ======================================================
# CONFIRMATION
# ======================================================

print("✔ Corpora loaded successfully.")

print("\nmy_corpus (all texts):")
print(f"  texts:     {len(set(s.metadata['source_filename'] for s in my_corpus))}")
print(f"  sentences: {len(my_corpus)}")

print(f"\nsyntactically_annotated_corpus (≥ {MIN_SENTENCES_SYN} sentences & 1 root only):")
print(f"  texts:     {len(set(s.metadata['source_filename'] for s in syntactically_annotated_corpus))}")
print(f"  sentences: {len(syntactically_annotated_corpus)}")


# ======================================================
# SHOW CLEANED SOURCE FILENAMES
# ======================================================

annotated_files = sorted(
    set(s.metadata["source_filename"] for s in syntactically_annotated_corpus)
)

print("\n📄 Source files in the syntactically annotated corpus:")
for fname in annotated_files:
    print("-", fname)



--- Loading corpora ---
✔ Corpora loaded successfully.

my_corpus (all texts):
  texts:     42
  sentences: 35096

syntactically_annotated_corpus (≥ 10 sentences & 1 root only):
  texts:     25
  sentences: 5786

📄 Source files in the syntactically annotated corpus:
- AOD-K20
- DD-K35
- DD-TD4a
- DMX-K43a
- DMX-L19
- Dk3-B
- Dk5-B
- Dk6-B
- Dk7-B
- GA-K20
- GBd-DH
- GBd-TD1
- MHD-MHDC
- MYF-K20
- NM-K35
- NM-TD4a
- PVr-K7a
- PY-Pt4
- RThQA-TD2
- RĀF-TD2
- WZ_K35
- ZPJ̌-TD2
- ZWY-K20
- ZWY-K43a
- ŠnŠ-K20


In [47]:
from collections import Counter

# ======================================================
# SENTENCE COUNTS PER FILE (SYNTACTIC CORPUS)
# ======================================================

sentences_per_file = Counter(
    s.metadata["source_filename"]
    for s in syntactically_annotated_corpus
)

print("\n📊 Sentence counts per file (syntactically annotated):")
for fname, count in sorted(sentences_per_file.items()):
    print(f"{fname}: {count}")



📊 Sentence counts per file (syntactically annotated):
AOD-K20: 67
DD-K35: 236
DD-TD4a: 1
DMX-K43a: 379
DMX-L19: 234
Dk3-B: 1
Dk5-B: 424
Dk6-B: 2
Dk7-B: 153
GA-K20: 28
GBd-DH: 1
GBd-TD1: 1680
MHD-MHDC: 243
MYF-K20: 133
NM-K35: 269
NM-TD4a: 35
PVr-K7a: 1
PY-Pt4: 278
RThQA-TD2: 19
RĀF-TD2: 198
WZ_K35: 1028
ZPJ̌-TD2: 2
ZWY-K20: 184
ZWY-K43a: 188
ŠnŠ-K20: 2


In [48]:
# excluding texts with less than 10 sentences for this specific query
from collections import defaultdict, Counter
import csv


def export_full_pos_to_csv(corpus, output_filename):
    # --------------------------------------------------
    # storage
    # --------------------------------------------------
    text_stats = defaultdict(Counter)
    sentence_counts = Counter()

    # keeps track of all ADV feature-value columns
    adv_feat_keys = set()

    # --------------------------------------------------
    # collect statistics
    # --------------------------------------------------
    for sentence in corpus:
        text_id = sentence.metadata.get("source_filename", "unknown")

        # count sentences per text
        sentence_counts[text_id] += 1

        for token in sentence.get_tokens():
            upos = token.upos

            # total tokens (base for density)
            if upos not in ("PUNCT", "X"):
                text_stats[text_id]["total_tokens"] += 1

            # POS categories
            if upos in {"NOUN", "PROPN"}:
                text_stats[text_id]["NOUN"] += 1

            elif upos == "PRON":
                text_stats[text_id]["PRON"] += 1

            elif upos == "ADJ":
                text_stats[text_id]["ADJ"] += 1

            elif upos == "ADV":
                text_stats[text_id]["ADV"] += 1

                # ------------------------------------------
                # ADV feature-value classification
                # ------------------------------------------
                feats = token.feats or {}
                for feat_key, feat_val in feats.items():
                    # handle multi-valued features (e.g. Degree=Cmp,Pos)
                    for val in str(feat_val).split(","):
                        col_name = f"ADV_{feat_key}_{val}"
                        text_stats[text_id][col_name] += 1
                        adv_feat_keys.add(col_name)

            elif upos == "VERB":
                text_stats[text_id]["VERB"] += 1

            elif upos == "DET":
                text_stats[text_id]["DET"] += 1

    # --------------------------------------------------
    # write CSV
    # --------------------------------------------------
    adv_feat_columns = sorted(adv_feat_keys)

    with open(output_filename, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)

        writer.writerow([
            "text_id",
            "sentences",
            "total_tokens",
            "NOUN",
            "PRON",
            "ADJ",
            "ADV",
            "VERB",
            "DET",
            *adv_feat_columns
        ])

        for tid in sorted(text_stats):
            s = text_stats[tid]
            writer.writerow([
                tid,
                sentence_counts[tid],
                s["total_tokens"],
                s["NOUN"],
                s["PRON"],
                s["ADJ"],
                s["ADV"],
                s["VERB"],
                s["DET"],
                *[s.get(col, 0) for col in adv_feat_columns]
            ])


# --------------------------------------------------
# run
# --------------------------------------------------
export_full_pos_to_csv(my_corpus, "total_corpus_full_pos.csv")
export_full_pos_to_csv(syntactically_annotated_corpus, "syn_corpus_full_pos.csv")

print("✔ CSV files exported successfully.")


✔ CSV files exported successfully.


In [49]:
import csv
from collections import defaultdict

# ==================================================
# STEP 1: Assign sentence numbers PER TEXT
# ==================================================
def assign_sentence_numbers_per_text(corpus):
    """
    Assign sentence_id = 1,2,3... per text (source_filename).
    """
    counters = defaultdict(int)

    for sentence in corpus:
        text_id = sentence.metadata.get("source_filename", "unknown")
        counters[text_id] += 1
        sentence.metadata["sentence_id"] = counters[text_id]


assign_sentence_numbers_per_text(my_corpus)
assign_sentence_numbers_per_text(syntactically_annotated_corpus)

# ==================================================
# STEP 2: Allowed Middle Persian ADV features
# ==================================================
ALLOWED_ADVTYPE = {"Man", "Loc", "Tim", "Deg", "Emp"}
ALLOWED_NUMTYPE = {"Mult", "Ord"}
ALLOWED_DEGREE  = {"Cmp", "Sup"}

# ==================================================
# STEP 3: Find wrongly annotated ADV tokens
# ==================================================
def find_wrongly_annotated_adv(corpus, output_csv):
    rows = []

    for sentence in corpus:
        text_id = sentence.metadata.get("source_filename", "unknown")
        sentence_id = sentence.metadata.get("sentence_id")

        for token in sentence.get_tokens():

            # only ADV
            if token.upos != "ADV":
                continue

            feats = dict(token.feats or {})

            # skip ADV with no features
            if not feats:
                continue

            # skip Typo=Yes anywhere
            if feats.get("Typo") == "Yes":
                continue

            advtype = (feats.get("AdvType") or "").strip().title()
            numtype = (feats.get("NumType") or "").strip()
            degree  = (feats.get("Degree") or "").strip()
            deixis  = (feats.get("Deixis") or "").strip()
            lemmata = token.lemma or ""

            # ✅ CORRECT sense extraction
            sense = getattr(token, "sense", "") or ""

            allowed = False

            # -------------------------------
            # Allowed adverbials
            # -------------------------------
            if advtype in ALLOWED_ADVTYPE:
                if advtype == "Loc":
                    if not deixis or deixis in {"Prox", "Remt"}:
                        allowed = True
                else:
                    allowed = True

            elif numtype in ALLOWED_NUMTYPE:
                allowed = True

            elif degree in ALLOWED_DEGREE:
                allowed = True

            # -------------------------------
            # Record wrongly annotated ADV
            # -------------------------------
            if not allowed:
                rows.append([
                    text_id,
                    sentence_id,
                    lemmata,
                    token.upos,
                    feats,
                    sense
                ])

    # ==================================================
    # Write CSV
    # ==================================================
    with open(output_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "text_id",
            "sentence_id",
            "lemmata",
            "upos",
            "feats",
            "sense"
        ])
        writer.writerows(rows)

    print(f"✔ Found {len(rows)} wrongly annotated ADV tokens → {output_csv}")

# ==================================================
# STEP 4: Run
# ==================================================
find_wrongly_annotated_adv(
    my_corpus,
    "wrongly_annotated_adv_tokens.csv"
)

find_wrongly_annotated_adv(
    syntactically_annotated_corpus,
    "wrongly_annotated_adv_tokens_syn.csv"
)


✔ Found 44 wrongly annotated ADV tokens → wrongly_annotated_adv_tokens.csv
✔ Found 14 wrongly annotated ADV tokens → wrongly_annotated_adv_tokens_syn.csv


In [50]:
# --------------------------------------------------
# GET ALL ADVERB ATTESTATIONS FOR A SPECIFIC FEAT
# --------------------------------------------------
def get_adv_attestations(corpus, feat_key, feat_val=None):
    """
    Collect all adverbs in the corpus with a specific feature key,
    optionally filtered by a specific value.

    Args:
        corpus: list of Sentence objects (from your mptf_parser)
        feat_key: str, the key in token.feats (e.g., "AdvType", "Degree")
        feat_val: str or None, optional. If given, only return tokens with this value

    Returns:
        List of tuples: (text_id, sentence_id, token_index, token_lemma, feats_dict)
    """
    attestations = []

    for sentence in corpus:
        text_id = sentence.file_name
        sent_id = sentence.sentence_id

        for t_idx, token in enumerate(sentence.get_tokens()):
            if token.upos != "ADV":
                continue

            feats = token.feats or {}
            val = feats.get(feat_key)
            if val is None:
                continue

            # handle multi-valued features like "Cmp,Pos"
            val_list = str(val).split(",")

            if feat_val is None or feat_val in val_list:
                lemma = token.lemma  # lemma is stored in Token.lemma
                attestations.append(
                    (text_id, sent_id, t_idx, lemma, feats)
                )

    return attestations


# Load your syntactically annotated corpus
# corpus = syntactically_annotated_corpus  # list of Sentence objects

# 1. All adverbs with any AdvType
all_advtype = get_adv_attestations(corpus, "AdvType")
print(f"Found {len(all_advtype)} adverbs with AdvType.")

# 2. Only adverbs with AdvType=Mod
searched_advs = get_adv_attestations(corpus, "VerbForm", "Part")
print(f"Found {len(searched_advs)} Mod adverbs.")

# 3. Print a few
for att in searched_advs:
    print(att)
    # output: ('filename.conllu', 'sent_23', 5, 'lemma_of_token', {'AdvType': 'Mod', ...})



Found 39461 adverbs with AdvType.
Found 13 Mod adverbs.
('MHD-MHDC_mptf.conllu', '56', 34, 'zīwandagān', {'AdvType': 'Tim', 'Number': 'Plur', 'Subcat': 'Intr', 'Tense': 'Pres', 'VerbForm': 'Part'})
('MHD-MHDC_mptf.conllu', '205', 14, 'zīwandagān', {'AdvType': 'Tim', 'Number': 'Plur', 'Subcat': 'Intr', 'Tense': 'Pres', 'VerbForm': 'Part'})
('MHD-MHDC_mptf.conllu', '262', 9, 'zīwandag', {'AdvType': 'Tim', 'Subcat': 'Intr', 'Tense': 'Pres', 'VerbForm': 'Part'})
('MHD-MHDC_mptf.conllu', '268', 9, 'zīwandag', {'AdvType': 'Tim', 'Subcat': 'Intr', 'Tense': 'Pres', 'VerbForm': 'Part'})
('MHD-MHDC_mptf.conllu', '581', 5, 'zīwandag', {'AdvType': 'Tim', 'Subcat': 'Intr', 'Tense': 'Pres', 'VerbForm': 'Part'})
('MHD-MHDC_mptf.conllu', '589', 1, 'zīwandagān', {'AdvType': 'Tim', 'Number': 'Plur', 'Subcat': 'Intr', 'Tense': 'Pres', 'VerbForm': 'Part'})
('MHD-MHDC_mptf.conllu', '736', 1, 'zīwandag', {'AdvType': 'Tim', 'Subcat': 'Intr', 'Tense': 'Pres', 'VerbForm': 'Part'})
('MHD-MHDC_mptf.conllu', '829

In [51]:
# --------------------------------------------------
# COLLECT TOKENS ENDING WITH ag OR agān
# --------------------------------------------------
def collect_tokens_ag_agān(corpus):
    """
    Collect tokens whose lemma or form ends with 'ag' or 'agān'.
    
    Returns:
        List of tuples: (text_id, sentence_id, token_index, token_form, token_lemma, upos, feats_dict)
    """
    matches = []

    for sentence in corpus:
        text_id = sentence.file_name
        sent_id = sentence.sentence_id

        for t_idx, token in enumerate(sentence.get_tokens()):
            token_lemma = token.lemma or ""
            token_form = token.form or ""
            token_feats = token.feats or {}
            if (token_lemma.endswith(("ag", "agān")) or token_form.endswith(("ag", "agān"))) and token.upos == "ADV" and token_feats.get("AdvType") == "Tim":
                matches.append(
                    (text_id, sent_id, t_idx, token_form, token_lemma, token.upos, token_feats)
                )

    return matches


# corpus = syntactically_annotated_corpus
ag_tokens = collect_tokens_ag_agān(corpus)
print(f"Found {len(ag_tokens)} tokens ending with 'ag' or 'agān'.")

# Print
for t in ag_tokens:
    print(t)
    # output: ('filename.conllu', 'sent_12', 5, 'form', 'lemma', 'ADV', {'AdvType': 'Mod', ...})



Found 137 tokens ending with 'ag' or 'agān'.
('AOD-K20_mptf.conllu', '91', 4, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('AOD-K20_mptf.conllu', '92', 6, 'hamwārag', 'hamwārag', 'ADV', {'AdvType': 'Tim'})
('DD-K35_mptf.conllu', '338', 6, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-K35_mptf.conllu', '466', 5, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-K35_mptf.conllu', '517', 23, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-K35_mptf.conllu', '814', 8, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-K35_mptf.conllu', '860', 2, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-TD4a_mptf.conllu', '343', 4, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-TD4a_mptf.conllu', '468', 5, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-TD4a_mptf.conllu', '518', 20, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-TD4a_mptf.conllu', '813', 8, 'hamēšag', 'hamēšag', 'ADV', {'AdvType': 'Tim'})
('DD-TD4a_mptf.conllu', '859', 2, 'hamēšag', 

In [52]:
# --------------------------------------------------
# COLLECT UNIQUE LEMMATA AND SENTENCE IDS
# --------------------------------------------------
from collections import defaultdict

def collect_unique_lemmata_with_sentences(corpus):
    """
    Collect unique ADV lemmata ending with 'ag' or 'agān' and AdvType='Tim',
    and aggregate all sentence IDs where each lemma appears.

    Returns:
        dict: {lemma: set of sentence_ids}
    """
    lemma_sent_ids = defaultdict(set)

    for sentence in corpus:
        sent_id = sentence.sentence_id

        for token in sentence.get_tokens():
            token_lemma = token.lemma or ""
            token_form = token.form or ""
            token_feats = token.feats or {}

            # Filter for ADV, ending with 'ag'/'agān', AdvType=Tim
            if token.upos == "ADV" \
                and token_feats.get("AdvType") == "Tim" \
                and (token_lemma.endswith(("ag", "agān")) or token_form.endswith(("ag", "agān"))):
                
                lemma_sent_ids[token_lemma].add(sent_id)

    return dict(lemma_sent_ids)  # convert to normal dict


# corpus = syntactically_annotated_corpus
lemma_dict = collect_unique_lemmata_with_sentences(corpus)

print(f"Found {len(lemma_dict)} unique temporal adverb lemmata ending with 'ag' or 'agān'.\n")

# Example output
for lemma, sent_ids in lemma_dict.items():
    print(f"Lemma: {lemma} | Sentences: {sorted(sent_ids)}")


Found 19 unique temporal adverb lemmata ending with 'ag' or 'agān'.

Lemma: hamēšag | Sentences: ['1296', '1297', '1304', '1408', '1539', '189', '190', '191', '2236', '226', '2316', '233', '234', '2391', '254', '261', '2782', '3065', '316', '322', '326', '328', '338', '343', '391', '402', '428', '466', '468', '488', '517', '518', '619', '679', '71', '750', '757', '768', '785', '813', '814', '859', '860', '907', '91', '969', '976', '984']
Lemma: hamwārag | Sentences: ['92']
Lemma: sešabag | Sentences: ['114', '231', '269', '412', '415', '637']
Lemma: zīwandagān | Sentences: ['106', '111', '120', '1216', '129', '1487', '1488', '155', '173', '176', '179', '182', '205', '25', '35', '37', '40', '44', '49', '54', '56', '589', '59', '67', '70', '73', '80', '829', '83', '95']
Lemma: zīwandag | Sentences: ['1485', '262', '268', '57', '581', '736']
Lemma: ǰāwēdānag | Sentences: ['352']
Lemma: ēwrōzag | Sentences: ['848']
Lemma: murdagān | Sentences: ['106', '111', '120', '129', '155', '176', '17